# 00f — Indian Classical Transcription (Colab T4)

Transcribes the remaining Hindustani and Carnatic tracks that were skipped in the
original local prep notebooks (which capped at 60 each due to CPU time).

**Run this on a Colab T4 GPU — Basic-Pitch inference is ~8-10x faster than CPU.**

| Tradition | Already done | Available | New to transcribe | Est. time (T4) |
|---|---|---|---|---|
| Hindustani | 60 | 108 | 48 | ~25 min |
| Carnatic | 60 | 149 (concert mixes) | 89 | ~45 min |

New MIDI files are saved directly to your Drive and appended to the existing
metadata CSVs. The training notebooks pick them up automatically on the next run.

**Note:** Carnatic tracks filtered to concert mixes only (files whose stem matches
their parent directory name). Separated stems (vocal-only, violin-only) are skipped.

## Cell 1 — Install Basic-Pitch

In [8]:
import os, subprocess, sys
import numpy as np

ON_COLAB = 'COLAB_GPU' in os.environ or 'COLAB_RELEASE_TAG' in os.environ
print(f'Environment: {"Colab" if ON_COLAB else "Local"}')

if ON_COLAB:
    # Colab-specific workarounds:
    # 1. numpy pip metadata is broken — force-reinstall same version to fix it
    # 2. librosa 0.11 (pre-installed) needs resampy>=0.4.3; basic-pitch declares <0.4.3
    #    → pip resolver deadlocks; bypass with --no-deps and manual dep installs
    print(f'Re-registering numpy {np.__version__}...')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
                           f'numpy=={np.__version__}', '--force-reinstall', '--no-build-isolation'])
    ip = get_ipython()
    ip.system('pip install "pretty-midi>=0.2.9" mir_eval resampy --no-build-isolation -q')
    ip.system('pip install "basic-pitch==0.4.0" --no-deps -q')
else:
    # resampy 0.4.2 (in the venv) imports pkg_resources which was removed from
    # setuptools 80+. resampy 0.4.3 switched to importlib.resources — upgrade it.
    # On Mac, basic-pitch uses CoreML (built into macOS) so no extra backend needed.
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'resampy>=0.4.3'])
    print('Local deps ready.')

import torch
print('CUDA:', torch.cuda.is_available())

from basic_pitch.inference import predict
from basic_pitch import ICASSP_2022_MODEL_PATH
print('Basic-Pitch ready:', ICASSP_2022_MODEL_PATH)

Environment: Local
Local deps ready.
CUDA: False
Basic-Pitch ready: /Users/mohammadashraf/Documents/GitHub/Thesis-Best/venv/lib/python3.11/site-packages/basic_pitch/saved_models/icassp_2022/nmp.mlpackage


## Cell 2 — Mount Drive and Set Paths

In [9]:
from pathlib import Path
import pandas as pd
import re, time, warnings
warnings.filterwarnings('ignore')

if ON_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE = Path('/content/drive/MyDrive/Thesis-Data')
    HIND_RAW  = BASE / 'datasets' / 'indian_classical' / 'saraga1.5_hindustani'
    CARN_RAW  = BASE / 'datasets' / 'indian_classical' / 'saraga1.5_carnatic'
    HIND_MIDI = BASE / 'data' / 'processed' / 'hindustani' / 'midi'
    CARN_MIDI = BASE / 'data' / 'processed' / 'carnatic'   / 'midi'
    META_DIR  = BASE / 'data' / 'metadata'
else:
    # All data lives in the repo — datasets/ and data/ are siblings of notebooks/
    REPO_ROOT = Path(__file__).resolve().parent.parent.parent if '__file__' in dir() else Path.cwd()
    # Walk up until we find the repo root (contains datasets/)
    while not (REPO_ROOT / 'datasets').exists() and REPO_ROOT != REPO_ROOT.parent:
        REPO_ROOT = REPO_ROOT.parent
    print(f'Repo root: {REPO_ROOT}')
    HIND_RAW  = REPO_ROOT / 'datasets' / 'indian_classical' / 'saraga1.5_hindustani'
    CARN_RAW  = REPO_ROOT / 'datasets' / 'indian_classical' / 'saraga1.5_carnatic'
    HIND_MIDI = REPO_ROOT / 'data' / 'processed' / 'hindustani' / 'midi'
    CARN_MIDI = REPO_ROOT / 'data' / 'processed' / 'carnatic'   / 'midi'
    META_DIR  = REPO_ROOT / 'data' / 'metadata'

HIND_MIDI.mkdir(parents=True, exist_ok=True)
CARN_MIDI.mkdir(parents=True, exist_ok=True)

HIND_META = META_DIR / 'hindustani_tracks.csv'
CARN_META = META_DIR / 'carnatic_tracks.csv'

print(f'Hindustani raw : {HIND_RAW.exists()}  {HIND_RAW}')
print(f'Carnatic raw   : {CARN_RAW.exists()}  {CARN_RAW}')
print(f'Hind MIDI dir  : {len(list(HIND_MIDI.glob("*.mid")))} files')
print(f'Carn MIDI dir  : {len(list(CARN_MIDI.glob("*.mid")))} files')

Repo root: /Users/mohammadashraf/Documents/GitHub/Thesis-Best
Hindustani raw : True  /Users/mohammadashraf/Documents/GitHub/Thesis-Best/datasets/indian_classical/saraga1.5_hindustani
Carnatic raw   : True  /Users/mohammadashraf/Documents/GitHub/Thesis-Best/datasets/indian_classical/saraga1.5_carnatic
Hind MIDI dir  : 60 files
Carn MIDI dir  : 60 files


## Cell 3 — Shared Transcription Helper

In [10]:
from basic_pitch.inference import predict
from basic_pitch import ICASSP_2022_MODEL_PATH
from concurrent.futures import ThreadPoolExecutor, TimeoutError as FuturesTimeout

def transcribe_audio(audio_path: Path, out_path: Path) -> bool:
    try:
        _, midi_data, _ = predict(str(audio_path), ICASSP_2022_MODEL_PATH)
        midi_data.write(str(out_path))
        return True
    except Exception as e:
        print(f'    ERROR: {e}')
        return False


def transcribe_with_timeout(audio_path: Path, out_path: Path, timeout: int = 180) -> bool:
    """Transcribe with a per-file timeout so a single hung file doesn't kill the session."""
    with ThreadPoolExecutor(max_workers=1) as ex:
        future = ex.submit(transcribe_audio, audio_path, out_path)
        try:
            return future.result(timeout=timeout)
        except FuturesTimeout:
            print(f'    TIMEOUT after {timeout}s — skipping')
            if out_path.exists():
                out_path.unlink()  # remove partial file
            return False


def safe_stem(path: Path) -> str:
    name = path.name
    for ext in ['.mp3.mp3', '.mp3', '.wav']:
        if name.endswith(ext):
            name = name[:-len(ext)]
            break
    return re.sub(r'[^\w\-]', '_', name)[:60]


def is_concert_mix(audio_path: Path) -> bool:
    stem = audio_path.name
    for ext in ['.mp3.mp3', '.mp3', '.wav']:
        if stem.endswith(ext):
            stem = stem[:-len(ext)]
    return stem == audio_path.parent.name


print('Helpers ready.')

Helpers ready.


## Cell 4 — Transcribe Remaining Hindustani Tracks

Finds all audio files in the Saraga Hindustani directory, skips any whose
audio path already appears in `hindustani_tracks.csv`, and transcribes the rest.

In [11]:
import json

# ── All Hindustani audio files (sorted = deterministic enumeration) ───────────
all_hind_audio = sorted(
    list(HIND_RAW.rglob('*.mp3.mp3')) +
    [f for f in HIND_RAW.rglob('*.mp3') if not str(f).endswith('.mp3.mp3')]
)
print(f'Total audio files : {len(all_hind_audio)}')

# ── Resume detection: check which output files already exist on Drive ─────────
# Output filename is DETERMINISTIC (derived from audio stem, not a counter),
# so it's always the same name regardless of how many times we restart.
def hind_out_name(i: int, audio_path: Path) -> str:
    return f'hindustani_{i:03d}_{safe_stem(audio_path)}.mid'

already_done = {
    i for i, f in enumerate(all_hind_audio)
    if (HIND_MIDI / hind_out_name(i, f)).exists()
}
todo = [(i, f) for i, f in enumerate(all_hind_audio) if i not in already_done]
print(f'Already done      : {len(already_done)}')
print(f'To transcribe     : {len(todo)}')

# ── Transcribe ────────────────────────────────────────────────────────────────
succeeded, failed, skipped = 0, 0, 0

# Load existing metadata CSV for appending
hind_meta = pd.read_csv(HIND_META) if HIND_META.exists() else pd.DataFrame()
done_paths = set(hind_meta['audio_path'].dropna()) if 'audio_path' in hind_meta.columns else set()

for n, (i, audio_path) in enumerate(todo, 1):
    out_name = hind_out_name(i, audio_path)
    out_path = HIND_MIDI / out_name

    # Should not happen, but guard anyway
    if out_path.exists():
        print(f'[{n:3d}/{len(todo)}] SKIP (exists) {out_name}')
        skipped += 1
        continue

    t0 = time.time()
    print(f'[{n:3d}/{len(todo)}] {audio_path.name[:60]} ...', end=' ', flush=True)
    ok = transcribe_with_timeout(audio_path, out_path, timeout=180)
    elapsed = time.time() - t0

    if ok:
        # Extract raga from sidecar JSON
        raga = None
        json_path = audio_path.parent / (audio_path.parent.name + '.json')
        if json_path.exists():
            try:
                meta = json.loads(json_path.read_text())
                ragas = meta.get('raags') or meta.get('ragas') or []
                raga = ragas[0].get('name') if ragas else None
            except Exception:
                pass

        row = {'audio_path': str(audio_path), 'midi_path': str(out_path),
               'raga': raga, 'source': 'saraga_hindustani'}

        # Checkpoint: append row to CSV immediately so progress survives a timeout
        row_df = pd.DataFrame([row])
        row_df.to_csv(HIND_META, mode='a',
                      header=not HIND_META.exists() and len(hind_meta) == 0,
                      index=False)
        hind_meta = pd.concat([hind_meta, row_df], ignore_index=True)

        succeeded += 1
        print(f'done ({elapsed:.0f}s)')
    else:
        failed += 1
        print(f'FAILED ({elapsed:.0f}s)')

print(f'\n=== Hindustani done: {succeeded} new, {skipped} skipped, {failed} failed ===')
print(f'Total MIDI files : {len(list(HIND_MIDI.glob("*.mid")))}')

Total audio files : 108
Already done      : 0
To transcribe     : 108
[  1/108] Raag Basanti Kedar.mp3.mp3 ... Predicting MIDI for /Users/mohammadashraf/Documents/GitHub/Thesis-Best/datasets/indian_classical/saraga1.5_hindustani/Anaahata by Milind Malshe/Raag Basanti Kedar/Raag Basanti Kedar.mp3.mp3...
isfinite: True
shape: (1, 43844, 1)
dtype: float32
isfinite: True
shape: (1, 43844, 1)
dtype: float32
isfinite: True
shape: (1, 43844, 1)
dtype: float32
isfinite: True
shape: (1, 43844, 1)
dtype: float32
isfinite: True
shape: (1, 43844, 1)
dtype: float32
isfinite: True
shape: (1, 43844, 1)
dtype: float32
isfinite: True
shape: (1, 43844, 1)
dtype: float32
isfinite: True
shape: (1, 43844, 1)
dtype: float32
isfinite: True
shape: (1, 43844, 1)
dtype: float32
isfinite: True
shape: (1, 43844, 1)
dtype: float32
isfinite: True
shape: (1, 43844, 1)
dtype: float32
isfinite: True
shape: (1, 43844, 1)
dtype: float32
isfinite: True
shape: (1, 43844, 1)
dtype: float32
isfinite: True
shape: (1, 43844, 

Note: Illegal Audio-MPEG-Header 0x44d3a352 at offset 54923548.
Note: Trying to resync...
Note: Skipped 244 bytes in input.


isfinite: True
shape: (1, 43844, 1)
dtype: float32
isfinite: True
shape: (1, 43844, 1)
dtype: float32
isfinite: True
shape: (1, 43844, 1)
dtype: float32
isfinite: True
shape: (1, 43844, 1)
dtype: float32
isfinite: True
shape: (1, 43844, 1)
dtype: float32
isfinite: True
shape: (1, 43844, 1)
dtype: float32
isfinite: True
shape: (1, 43844, 1)
dtype: float32
isfinite: True
shape: (1, 43844, 1)
dtype: float32
isfinite: True
shape: (1, 43844, 1)
dtype: float32
isfinite: True
shape: (1, 43844, 1)
dtype: float32
isfinite: True
shape: (1, 43844, 1)
dtype: float32
isfinite: True
shape: (1, 43844, 1)
dtype: float32
isfinite: True
shape: (1, 43844, 1)
dtype: float32
isfinite: True
shape: (1, 43844, 1)
dtype: float32
isfinite: True
shape: (1, 43844, 1)
dtype: float32
isfinite: True
shape: (1, 43844, 1)
dtype: float32
isfinite: True
shape: (1, 43844, 1)
dtype: float32
isfinite: True
shape: (1, 43844, 1)
dtype: float32
isfinite: True
shape: (1, 43844, 1)
dtype: float32
isfinite: True
shape: (1, 43844

KeyboardInterrupt: 

## Cell 5 — Transcribe Remaining Carnatic Tracks

Saraga Carnatic contains both concert mixes and separated stems (vocal-only,
violin-only, etc.). We only want concert mixes — detected by checking whether
the audio file stem matches its parent directory name.

In [ ]:
# ── All Carnatic concert-mix audio files (sorted = deterministic) ─────────────
all_carn_audio = sorted(
    list(CARN_RAW.rglob('*.mp3.mp3')) +
    [f for f in CARN_RAW.rglob('*.mp3') if not str(f).endswith('.mp3.mp3')]
)
concert_mixes = [f for f in all_carn_audio if is_concert_mix(f)]
print(f'Total audio files : {len(all_carn_audio)}')
print(f'Concert mixes     : {len(concert_mixes)}')

def carn_out_name(i: int, audio_path: Path) -> str:
    return f'carnatic_{i:03d}_{safe_stem(audio_path)}.mid'

already_done = {
    i for i, f in enumerate(concert_mixes)
    if (CARN_MIDI / carn_out_name(i, f)).exists()
}
todo = [(i, f) for i, f in enumerate(concert_mixes) if i not in already_done]
print(f'Already done      : {len(already_done)}')
print(f'To transcribe     : {len(todo)}')

# ── Transcribe ────────────────────────────────────────────────────────────────
succeeded, failed, skipped = 0, 0, 0

carn_meta = pd.read_csv(CARN_META) if CARN_META.exists() else pd.DataFrame()

for n, (i, audio_path) in enumerate(todo, 1):
    out_name = carn_out_name(i, audio_path)
    out_path = CARN_MIDI / out_name

    if out_path.exists():
        print(f'[{n:3d}/{len(todo)}] SKIP (exists) {out_name}')
        skipped += 1
        continue

    t0 = time.time()
    print(f'[{n:3d}/{len(todo)}] {audio_path.name[:60]} ...', end=' ', flush=True)
    ok = transcribe_with_timeout(audio_path, out_path, timeout=180)
    elapsed = time.time() - t0

    if ok:
        raaga = None
        json_path = audio_path.parent / (audio_path.parent.name + '.json')
        if json_path.exists():
            try:
                meta = json.loads(json_path.read_text())
                raagas = meta.get('raaga') or meta.get('ragas') or []
                raaga = raagas[0].get('name') if raagas else None
            except Exception:
                pass

        row = {'audio_path': str(audio_path), 'midi_path': str(out_path),
               'raga': raaga, 'source': 'saraga_carnatic'}

        # Checkpoint: write each row immediately so restarts pick up where we left off
        row_df = pd.DataFrame([row])
        row_df.to_csv(CARN_META, mode='a',
                      header=not CARN_META.exists() and len(carn_meta) == 0,
                      index=False)
        carn_meta = pd.concat([carn_meta, row_df], ignore_index=True)

        succeeded += 1
        print(f'done ({elapsed:.0f}s)')
    else:
        failed += 1
        print(f'FAILED ({elapsed:.0f}s)')

print(f'\n=== Carnatic done: {succeeded} new, {skipped} skipped, {failed} failed ===')
print(f'Total MIDI files : {len(list(CARN_MIDI.glob("*.mid")))}')

Total audio files : 0
Concert mixes     : 0
Already done      : 0
To transcribe     : 0

=== Carnatic done: 0 new, 0 skipped, 0 failed ===
Total MIDI files : 0


## Cell 6 — Final Summary

In [ ]:
print('=== Indian Classical Transcription Summary ===')
print()

for tradition, midi_dir, meta_path in [
    ('Hindustani', HIND_MIDI, HIND_META),
    ('Carnatic',   CARN_MIDI, CARN_META),
]:
    midi_files = list(midi_dir.glob('*.mid'))
    meta       = pd.read_csv(meta_path) if meta_path.exists() else pd.DataFrame()
    with_midi  = meta['midi_path'].notna().sum() if 'midi_path' in meta.columns else 0
    print(f'{tradition}')
    print(f'  MIDI files in processed dir : {len(midi_files)}')
    print(f'  Rows in metadata CSV        : {len(meta)}')
    print(f'  Rows with midi_path         : {with_midi}')
    print()

print('Next step: re-run the Music Transformer Colab notebook.')
print('The training scripts will automatically pick up all MIDI files in the processed dirs.')

=== Indian Classical Transcription Summary ===

Hindustani
  MIDI files in processed dir : 0
  Rows in metadata CSV        : 0
  Rows with midi_path         : 0

Carnatic
  MIDI files in processed dir : 0
  Rows in metadata CSV        : 0
  Rows with midi_path         : 0

Next step: re-run the Music Transformer Colab notebook.
The training scripts will automatically pick up all MIDI files in the processed dirs.
